In [1]:
# Import thư viện và module nội bộ
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
# Thêm đường dẫn tới thư mục src (tùy cấu trúc workspace)
sys.path.append(os.path.abspath(os.path.join('..','src')))
from apriori_library import (
    DataCleaner,
    BasketPreparer,
    AssociationRulesMiner,
    FPGrowthMiner,
    benchmark_apriori_vs_fpgrowth,
    WeightedRulesAugmenter,
)

In [2]:
# Load dữ liệu đã xử lý (đảm bảo file tồn tại)
data_path = os.path.abspath(os.path.join('..','data','processed','cleaned_uk_data.csv'))
df = pd.read_csv(data_path, parse_dates=['InvoiceDate'])
print('Loaded:', data_path)
df.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_28012\882151293.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, parse_dates=['InvoiceDate'])


Loaded: c:\Code\DataMining\shopping_cart_advanced_analysis\data\processed\cleaned_uk_data.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,DayOfWeek,HourOfDay
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,017850,United Kingdom,15.30,2,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,017850,United Kingdom,20.34,2,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,017850,United Kingdom,22.00,2,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,017850,United Kingdom,20.34,2,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,017850,United Kingdom,20.34,2,8


In [3]:
# Chuẩn bị basket boolean
bp = BasketPreparer(df, invoice_col='InvoiceNo', item_col='Description', quantity_col='Quantity')
basket = bp.create_basket()
basket_bool = bp.encode_basket(threshold=1)
print('Số hóa đơn (rows):', basket_bool.shape[0], 'Số sản phẩm (cols):', basket_bool.shape[1])
basket_bool.iloc[:5,:10]

Số hóa đơn (rows): 18021 Số sản phẩm (cols): 4007


Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT
InvoiceNo,,,,,,,,,,
536365,False,False,False,False,False,False,False,False,False,False
536366,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,False,False,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False


In [ ]:
# Thử nghiệm benchmark cho nhiều giá trị min_support
min_supports = [0.05, 0.03, 0.02, 0.01, 0.005]
summaries = []
for s in min_supports:
    res = benchmark_apriori_vs_fpgrowth(basket_bool, min_support=s, max_len=None, metric='lift', min_threshold=1.0)
    df_sum = res['summary'].copy()
    df_sum['min_support'] = s
    summaries.append(df_sum)
summary_all = pd.concat(summaries, ignore_index=True)
# pivot để dễ so sánh
display(summary_all.sort_values(['min_support','algorithm']))

In [ ]:
# Vẽ các đồ thị so sánh (thời gian, số itemset, số luật)
plt.figure(figsize=(10,5))
for metric in ['runtime_sec','n_itemsets','n_rules','avg_itemset_length']:
    plt.clf()
    for alg in ['apriori','fpgrowth']:
        sub = summary_all[summary_all['algorithm']==alg]
        plt.plot(sub['min_support'], sub[metric], marker='o', label=alg)
    plt.gca().invert_xaxis()
    plt.xlabel('min_support (giảm dần)')
    plt.ylabel(metric)
    plt.title(f'{metric} theo min_support')
    plt.legend()
    plt.show()

## So sánh luật thông thường và luật có trọng số (ví dụ với min_support=0.01)
Chọn một ngưỡng để minh hoạ cách tính weighted metrics và so sánh top luật theo `lift` và `weighted_lift`.

In [ ]:
# Chạy miners với min_support = 0.01
min_s = 0.01
ap = AssociationRulesMiner(basket_bool)
fi_ap = ap.mine_frequent_itemsets(min_support=min_s)
rules_ap = ap.generate_rules(metric='lift', min_threshold=1.0)
rules_ap = ap.add_readable_rule_str()
fp = FPGrowthMiner(basket_bool)
fi_fp = fp.mine_frequent_itemsets(min_support=min_s)
rules_fp = fp.generate_rules(metric='lift', min_threshold=1.0)
rules_fp = fp.add_readable_rule_str()
print('Apriori rules:', len(rules_ap), 'FP-Growth rules:', len(rules_fp))
rules_ap.head(5)

In [ ]:
# Tạo augmenter và thêm các cột weighted_* vào bảng luật Apriori
augmenter = WeightedRulesAugmenter(df, invoice_col='InvoiceNo', item_col='Description', weight_col='TotalPrice')
rules_ap_w = augmenter.augment_rules(rules_ap)
# Hiển thị top 10 theo lift và theo weighted_lift để so sánh
top_lift = rules_ap_w.sort_values('lift', ascending=False).head(10)
top_wlift = rules_ap_w.sort_values('weighted_lift', ascending=False).head(10)
print('Top theo lift (thường):')
display(top_lift[[ 'rule_str','support','confidence','lift','weighted_support','weighted_confidence','weighted_lift']])
print('Top theo weighted_lift:')
display(top_wlift[[ 'rule_str','support','confidence','lift','weighted_support','weighted_confidence','weighted_lift']])

In [ ]:
# Lưu kết quả (tuỳ chọn)
out_dir = os.path.abspath(os.path.join('..','data','processed'))
os.makedirs(out_dir, exist_ok=True)
rules_ap_w.to_csv(os.path.join(out_dir, 'apriori_rules_weighted_example.csv'), index=False)
print('Đã lưu kết quả vào', out_dir)

## Gợi ý phân tích (viết bằng tiếng Việt)
- So sánh số lượng itemset và luật khi giảm `min_support`: thuật toán nào nhạy hơn?
- So sánh thời gian chạy; đặc biệt khi `min_support` nhỏ thì Apriori thường chậm hơn FP-Growth.
- So sánh độ dài trung bình của itemset và chất lượng luật: luật có trọng số có thể ưu tiên các tập xuất hiện trong các hoá đơn có doanh thu lớn hơn.
- Kết luận: viết nhận xét ngắn gọn về giới hạn và độ nhạy tham số cho từng thuật toán.